# Project Atlas — gsplat Training (Colab)

**Hybrid pipeline step.**  
Run COLMAP locally → upload output here → train Gaussian Splat on free T4 → download `.splat`

**Workflow:**
1. Run `capture.py` + COLMAP locally → zip `data/scans/{scan_id}/colmap_output/`
2. Upload the zip in Cell 3
3. Run all cells → get `{scan_id}_splat.zip` to download
4. Unzip into `data/scans/{scan_id}/splat/` locally

**Expected time:** ~10 min on free T4 for a single room (30–60 frames, 7000 iterations)

In [ ]:
# Cell 1 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to GPU runtime!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2 — Install gsplat
!pip install gsplat==1.3.0 -q
!pip install nerfstudio -q
import gsplat
print('gsplat version:', gsplat.__version__)

In [ ]:
# Cell 3 — Upload COLMAP output zip
# Upload: data/scans/{scan_id}/colmap_output/ zipped as colmap_output.zip
from google.colab import files
import zipfile, os

print('Upload your colmap_output.zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

os.makedirs('/content/colmap_output', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/colmap_output')

print('Extracted to /content/colmap_output')
!find /content/colmap_output -type f | head -20

In [ ]:
# Cell 4 — Convert COLMAP to nerfstudio format
import subprocess, os

colmap_dir = '/content/colmap_output'
ns_data_dir = '/content/ns_data'
os.makedirs(ns_data_dir, exist_ok=True)

result = subprocess.run([
    'ns-process-data', 'colmap',
    '--data', colmap_dir,
    '--output-dir', ns_data_dir,
], capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('ns-process-data failed')

print('Converted. Contents:')
!ls -la /content/ns_data/

In [ ]:
# Cell 5 — Train gsplat
import subprocess

output_dir = '/content/splat_output'

result = subprocess.run([
    'ns-train', 'splatfacto',
    '--data', '/content/ns_data',
    '--output-dir', output_dir,
    '--max-num-iterations', '7000',
    '--pipeline.model.sh-degree', '0',  # lower VRAM
], capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError('Training failed')

print('Training done.')
!find /content/splat_output -name '*.splat' -o -name '*.ckpt' | head -10

In [ ]:
# Cell 6 — Export to .splat (viewer-compatible)
import subprocess, glob

# Find the latest checkpoint
ckpts = glob.glob('/content/splat_output/**/nerfstudio_models/*.ckpt', recursive=True)
if not ckpts:
    raise FileNotFoundError('No checkpoint found. Did training complete?')

config_path = ckpts[0].replace('nerfstudio_models', '').replace('.ckpt', '').rsplit('/', 1)[0]
config_yaml = glob.glob(f'{config_path}/../config.yml')[0]

export_dir = '/content/splat_export'
subprocess.run([
    'ns-export', 'gaussian-splat',
    '--load-config', config_yaml,
    '--output-dir', export_dir,
], check=True)

print('Export done.')
!ls -lh /content/splat_export/

In [ ]:
# Cell 7 — Download result
import zipfile, os
from google.colab import files

zip_out = '/content/splat_result.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk('/content/splat_export'):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, '/content/splat_export'))

print('Downloading splat_result.zip...')
print('Unzip this into: data/scans/{scan_id}/splat/')
files.download(zip_out)